In [ ]:
import warnings
import torch
import torch.nn.functional as F
from torchvision.transforms import functional as FT
from PIL import Image
from einops import rearrange

from wan import WanI2V
from wan.modules.vae import WanVAE
from wan.modules.clip import CLIPModel
from wan.configs.wan_i2v_14B import i2v_14B

from torchcodec.decoders import VideoDecoder
import torchvision
from torchvision import transforms

warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")

/local_scratch/gzappavi/wan_experiments/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/local_scratch/gzappavi/wan_experiments/wan2.1/wan/modules/model.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)
/local_scratch/gzappavi/wan_experiments/wan2.1/wan/modules/model.py:42: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @amp.autocast(enabled=False)


In [2]:
# !mkdir -p ./weights
# !hf download Wan-AI/Wan2.1-I2V-14B-480P --local-dir ./weights/Wan2.1-I2V-14B-480P


In [3]:
wan_i2v = WanI2V(
    config=i2v_14B,
    checkpoint_dir="./weights/Wan2.1-I2V-14B-480P/",
    device_id=0,
    t5_cpu=True,
)

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 47.40 GiB of which 7.88 MiB is free. Process 390954 has 46.70 GiB memory in use. Including non-PyTorch memory, this process has 624.00 MiB memory in use. Of the allocated memory 312.69 MiB is allocated by PyTorch, and 7.31 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# prompt = "Two young women, dressed in summer dresses, are walking and conversing in a vast, verdant field. The woman on the left has long, dark brown hair and is wearing a flowing, off-the-shoulder red dress with white patterns, looking towards her companion and smiling. The woman on the right has lighter, possibly reddish-blonde hair and is wearing a white sleeveless dress with small dark polka dots, also smiling and looking at her friend; both are wearing white sneakers. A narrow, grassy path is visible between rows of what appear to be young green bushes or crops, possibly berry bushes, stretching far into the background, with the rows creating a strong sense of perspective, converging towards the horizon under an overcast sky that suggests a soft, diffused light, contributing to the overall natural, serene, and friendly atmosphere of this relaxed interaction in an open agricultural landscape."
img = Image.open("examples/women_looking_at_each_other.jpg").convert("RGB")

target_size = (480, 832)

transform = transforms.Compose(
    [transforms.Resize(min(target_size)), transforms.CenterCrop(target_size)]
)
img = transform(img)
img.save("examples/transformed.jpg", quality=95)


prompt = "Two smiling young women in summer dresses and white sneakers walk and converse in a vast, green field of uniform crop rows receding into the distance. The woman on the left wears a red, off-the-shoulder dress, while the woman on the right wears a white polka-dot dress. An overcast sky provides soft, diffused light over the serene agricultural landscape."
negative_prompt = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"

video = wan_i2v.generate(
    prompt,
    img,
    max_area=target_size[0] * target_size[1],
    # sampling_steps=40,
    sampling_steps=2,
    n_prompt=negative_prompt,
)

NameError: name 'wan_i2v' is not defined

In [ ]:
vae = WanVAE(vae_pth="./weights/Wan2.1-I2V-14B-480P/Wan2.1_VAE.pth", device="cuda")
clip = CLIPModel(
    dtype=torch.float16,
    device="cuda",
    checkpoint_path="./weights/Wan2.1-I2V-14B-480P/models_clip_open-clip-xlm-roberta-large-vit-huge-14.pth",
    tokenizer_path="./weights/Wan2.1-I2V-14B-480P/xlm-roberta-large",
)

In [ ]:
# Prepare image

img = Image.open("./examples/women_looking_at_each_other.jpg").convert("RGB")
img_tensor = FT.to_tensor(img)

h, w = target_size = (480, 832)  # (height, width)

resize_dim = max(target_size)
img_tensor = FT.resize(img_tensor, max(target_size))
img_tensor = FT.center_crop(img_tensor, target_size)

In [ ]:
patch_size = (1, 2, 2)
vae_stride = (4, 8, 8)
frame_num = 81


def build_mask(lat_h, lat_w, frame_num, repeats=4):
    msk = torch.zeros(repeats + (frame_num - 1), lat_h, lat_w, device="cuda")
    msk[:repeats] = 1

    msk = rearrange(msk, "(g r) h w -> r g h w", r=repeats)

    return msk


# aspect_ratio = h / w
# max_area = h * w

# lat_h = round(
#     np.sqrt(max_area * aspect_ratio) // vae_stride[1] // patch_size[1] * patch_size[1]
# )
# lat_w = round(
#     np.sqrt(max_area / aspect_ratio) // vae_stride[2] // patch_size[2] * patch_size[2]
# )
lat_h, lat_w = h // vae_stride[1], w // vae_stride[2]


# normalize between -1 and 1 (with 2 * (x - 0.5))
img_tensor = img_tensor.sub_(0.5).div_(0.5).to("cuda")
clip_context = clip.visual(rearrange(img_tensor, "c h w -> 1 c 1 h w"))

padding = torch.zeros(3, frame_num - 1, h, w, device=img_tensor.device)

padded_frame = torch.concat([img_tensor.unsqueeze(1), padding], dim=1)


y = vae.encode([padded_frame])[0]

msk = build_mask(lat_h, lat_w, frame_num)
y = torch.concat([msk, y])

In [ ]:
import torch
from diffusers import WanPipeline, AutoModel, AutoencoderKLWan

# in any case, this is the model we need to use
model_id = "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers"


# or we can use WanVAE from the Wan repo
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)

In [ ]:
# load video
# same approach as Unified loader from Dyffsynth?
# what about multitalk? What does it use? -> there's no training code!
# UPDATE: for now we only need to load the first frame!


# load bboxes list and turn to arr
# UPDATE: depending of the approach, we need only the
# build characters masks (and background?)
# resize them (by nearest interpolation) to match latent space dimension


# compute video tokens by passing it through the VAE?

# apply log to masks before passing them to `scaled_dot_product_attention`?
# what kind of normalization is necessary for the mask that we pass to `scaled_dot_product_attention`
# it seems like there was a sort of weighted sum in multitalk, to check

# eventually compute the hard masks with the argmax on the attention maps, as done in multitalk
# the first frame (the "conditioning frame") must always be prepended

# compute text tokens for "A is looking at B"
# what do we use for A and B?
# does it make sense to have a VLLM caption the the characters in the conditioning image?


In [ ]:
# Maybe load wan and pass the video through the VAE?

In [ ]:
F.scaled_dot_product_attention()